# Advanced SQL — PT Kirimin
Kita memisahkan tahap query dengan CTE dan menjaga seluruh event tetap terlihat saat memakai window function.

> Diverifikasi: PostgreSQL v18.6 — sumber: https://www.postgresql.org/docs/release/ — tanggal cek: 2026-09-18

In [ ]:
import sqlite3, pandas as pd
cx = sqlite3.connect(':memory:')
events = pd.DataFrame([
 ('O-1','created','2026-09-01 08:00:00'), ('O-1','picked_up','2026-09-01 10:00:00'), ('O-1','delivered','2026-09-02 12:00:00'),
 ('O-2','created','2026-09-01 09:00:00'), ('O-2','picked_up','2026-09-01 12:00:00'),
], columns=['order_id','status','event_time'])
events.to_sql('events', cx, index=False)

In [ ]:
latest = pd.read_sql_query('''
WITH ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY event_time DESC) AS rn
 FROM events
)
SELECT order_id, status, event_time FROM ranked WHERE rn = 1
''', cx)
latest

In [ ]:
timeline = pd.read_sql_query('''
SELECT order_id, status, event_time,
       LAG(status) OVER (PARTITION BY order_id ORDER BY event_time) AS previous_status,
       ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY event_time) AS sequence_no
FROM events
ORDER BY order_id, event_time
''', cx)
timeline

## Mini-exercise
1. Tambahkan `LAG(event_time)` dan hitung durasi antar event.
2. Buat ranking order berdasarkan jumlah event.
# TODO: implementasikan query Anda.

## Takeaway
CTE memecah logika, window function menjaga konteks baris, dan EXPLAIN membantu menghubungkan SQL dengan biaya runtime.